# Full.ipynb — 完整 Car+Motor+Pass 直接建拓樸 → 特徵化 → 訓練

流程：
1. `preprocess(target='全部')` 取完整資料（含時間），先依時間切成 70% train / 30% test。
2. 以 train-only Jenks 建立醫院距離、城鎮距離、鄰近人口三個類別型地理特徵；原始經緯度只用來計算距離，不直接進 Mapper 或模型。
3. 原事故欄位與三個地理特徵一起建 train-only one-hot 與 MCA(n=5) lens；test 只做欄位對齊與 `transform`。
4. 用 `overlap=3, interval=8`（CubicalCover n_intervals=8, overlap_frac=0.3）只在 train 建 Mapper 圖。
5. 從 train Mapper 提取 cycle、core、centrality、articulation、bridge 與距離特徵；test 只繼承最近 train observation 的拓樸摘要。
6. distance/coreness/centrality 類特徵使用 train-only Jenks；三個布林特徵保留 0/1。全程不使用死亡標籤，也不建立 occupancy matrix。


In [1]:
%load_ext autoreload
%autoreload 2
import os, pickle, time
import numpy as np
import pandas as pd
import networkx as nx
import prince

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
version3_path = os.path.join(parent_dir, "Version3")
os.chdir(version3_path)   # 借用 Version3 的 tdamapper / utils / Data

from sklearn.cluster import AgglomerativeClustering
from tdamapper.core_old import MapperAlgorithm
from tdamapper.cover import CubicalCover
from tdamapper.clustering import FailSafeClustering
from utils.preprocess import preprocess, process_other

# 載入新版 models.py（Version3/utils 沒有 models，用檔案路徑載）
import importlib.util
_mp = os.path.join(parent_dir, "Models", "utils", "models.py")
_spec = importlib.util.spec_from_file_location("models_new", _mp)
models_new = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(models_new)
_ev = os.path.join(parent_dir, "Models", "utils", "evaluate.py")
_spec2 = importlib.util.spec_from_file_location("evaluate", _ev)
evaluate = importlib.util.module_from_spec(_spec2); _spec2.loader.exec_module(evaluate)

dataA2 = pd.read_csv("./Data/A2.csv", low_memory=False)
dataA1 = pd.read_csv("./Data/A1.csv")

/var/folders/w2/_g9w5yys0f171q4qqm469z1h0000gn/T/ipykernel_67825/3915157001.py:5: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [ ]:
select_lst = [
    # 月份是為了篩選每個月2萬筆
    '發生月份',

    '天候名稱', '光線名稱', 
    '道路類別-第1當事者-名稱', '速限-第1當事者', 
    '路面狀況-路面鋪裝名稱', '路面狀況-路面狀態名稱', '路面狀況-路面缺陷名稱',
    '道路障礙-障礙物名稱', '道路障礙-視距品質名稱', '道路障礙-視距名稱',
    '號誌-號誌種類名稱', '號誌-號誌動作名稱',
    '車道劃分設施-分道設施-快車道或一般車道間名稱', '車道劃分設施-分道設施-快慢車道間名稱', '車道劃分設施-分道設施-路面邊線名稱',
    '當事者屬-性-別名稱', '當事者事故發生時年齡',
    '保護裝備名稱', '行動電話或電腦或其他相類功能裝置名稱',
    # '肇事逃逸類別名稱-是否肇逃',
    '死亡受傷人數',

    # 大類別
    '道路型態大類別名稱', '事故位置大類別名稱',
    '車道劃分設施-分向設施大類別名稱',
    '事故類型及型態大類別名稱', '當事者區分-類別-大類別名稱-車種', '當事者行動狀態大類別名稱',
    # '車輛撞擊部位大類別名稱-最初', '車輛撞擊部位大類別名稱-其他',
    
    # 子類別
    # '道路型態子類別名稱', '事故位置子類別名稱', '當事者行動狀態子類別名稱',
    # '事故類型及型態子類別名稱', '肇因研判子類別名稱-主要', '當事者行動狀態子類別名稱', '當事者區分-類別-子類別名稱-車種', 
    # '車輛撞擊部位子類別名稱-最初', '車輛撞擊部位子類別名稱-其他', '肇因研判子類別名稱-個別', '肇因研判子類別名稱-主要'
]

# OSM Jenks 是 preprocess 後才生成的欄位，會在後面加入 mapper_select_lst；原始經緯度不加入。
full_dataA1 = preprocess(dataA1, target='全部', lst=select_lst, add_features=True)
full_dataA2 = preprocess(dataA2, target='全部', lst=select_lst, add_features=True)
# add_features=True 會多出：對造車種/當事人數/有無大型車、時段/深夜/週末/季節（無洩漏）
mapper_numpy, rbind_data, dummy_data, death, injuried, date_info = process_other(
    full_dataA1, full_dataA2, downsample=False, en=False, return_time=True)
print('rbind_data:', rbind_data.shape, '| dummy:', dummy_data.shape)

In [ ]:
# ==== 表示學習先依時間切分：前 70% 建 MCA/Mapper，後 30% 是 OOS pool ====
# OOS pool 會在模型階段再依時間分成 10% validation + 20% final test。
TRAIN_FRAC = 0.70
VALIDATION_FRAC = 0.10
TEST_FRAC = 0.20
assert np.isclose(TRAIN_FRAC + VALIDATION_FRAC + TEST_FRAC, 1.0)

_n = len(rbind_data)
_order_index = date_info.sort_values(['發生日期', '發生時間']).index
_order_pos = rbind_data.index.get_indexer(_order_index)
if (_order_pos < 0).any():
    raise ValueError('date_info 與 rbind_data 的 index 無法對齊')
_n_train = int(np.floor(_n * TRAIN_FRAC))
_n_validation = int(np.floor(_n * VALIDATION_FRAC))
_validation_end = _n_train + _n_validation
train_pos = _order_pos[:_n_train]
validation_pos = _order_pos[_n_train:_validation_end]
final_test_pos = _order_pos[_validation_end:]
test_pos = _order_pos[_n_train:]  # Mapper OOS pool = validation + final test
assert len(np.unique(_order_pos)) == _n
assert np.intersect1d(train_pos, test_pos).size == 0
assert np.array_equal(np.sort(np.r_[train_pos, test_pos]), np.arange(_n))
assert np.array_equal(test_pos, np.r_[validation_pos, final_test_pos])
print('time split | train:', len(train_pos),
      '| validation:', len(validation_pos),
      '| final test:', len(final_test_pos),
      '| Mapper OOS pool:', len(test_pos))


In [ ]:
# ==== OSM 地理特徵：原始經緯度只用來算距離；所有補值與 Jenks 分界只用 train ====
import geopandas as gpd
from sklearn.neighbors import BallTree
import jenkspy

OSM_DIR = "/Users/wangqiqian/Desktop/QGIS/taiwan-251009-free.shp"
R_KM = 6371.0088

def _latlon_rad(gdf):
    g = gdf.copy()
    if g.crs is not None and g.crs.to_epsg() != 4326:   # 若非 WGS84 就轉（本資料已是 4326，不需 transform）
        g = g.to_crs(4326)
    pts = g.geometry.representative_point()             # 點層回傳點本身；面層回傳內部代表點
    return np.radians(np.c_[pts.y.values, pts.x.values])

# 醫院：點層 + 面層
_h1 = gpd.read_file(f"{OSM_DIR}/gis_osm_pois_free_1.shp");   _h1 = _h1[_h1['fclass'] == 'hospital']
_h2 = gpd.read_file(f"{OSM_DIR}/gis_osm_pois_a_free_1.shp"); _h2 = _h2[_h2['fclass'] == 'hospital']
hosp_rad = np.vstack([_latlon_rad(_h1), _latlon_rad(_h2)])

# 城鎮（city/town）與 有人口的地點
_places = gpd.read_file(f"{OSM_DIR}/gis_osm_places_free_1.shp")
city_rad = _latlon_rad(_places[_places['fclass'].isin(['city', 'town'])])
_pop = _places[_places['population'] > 0]
pop_rad = _latlon_rad(_pop)
pop_val = _pop['population'].to_numpy()

# 事故座標（來自 date_info，與 rbind_data 對齊）；濾掉台灣範圍外的錯誤座標
lat = pd.to_numeric(date_info['緯度'], errors='coerce').to_numpy()
lon = pd.to_numeric(date_info['經度'], errors='coerce').to_numpy()
valid = np.isfinite(lat) & np.isfinite(lon) & (lat > 20) & (lat < 26.5) & (lon > 118) & (lon < 122.5)
acc_rad = np.radians(np.c_[np.where(valid, lat, 0.0), np.where(valid, lon, 0.0)])

def _nearest(tree_pts):
    d, i = BallTree(tree_pts, metric='haversine').query(acc_rad, k=1)
    return d[:, 0] * R_KM, i[:, 0]

d_h, _   = _nearest(hosp_rad)     # 最近醫院距離(km) = 送醫可及性
d_c, _   = _nearest(city_rad)     # 最近城鎮距離(km) = 城鄉
d_p, i_p = _nearest(pop_rad)      # 最近有人口地點 → 取其人口

spatial_raw = pd.DataFrame({
    'dist_hospital_band': np.where(valid, d_h, np.nan),
    'dist_city_band':     np.where(valid, d_c, np.nan),
    'nearest_pop_band':   np.where(valid, pop_val[i_p], np.nan),
}, index=rbind_data.index)
print(spatial_raw.describe().T[['mean', '50%', 'max']])

def jenks_bin_train(series, train_positions, n_classes=5, sample=20000, random_state=42):
    """只用 train 補值並估 Jenks 內部分界，再套用到全體；輸出皆為類別字串。"""
    v = pd.to_numeric(series, errors='coerce')
    tr = v.iloc[train_positions].dropna()
    if tr.empty:
        raise ValueError(f'{series.name} 的 train 資料全為缺值')
    v = v.fillna(tr.median())
    tr = v.iloc[train_positions]
    k = min(n_classes, tr.nunique())
    if k < 2:
        return pd.Series('L1', index=v.index, dtype=object)
    s = tr.sample(min(len(tr), sample), random_state=random_state) if len(tr) > sample else tr
    try:
        breaks = jenkspy.jenks_breaks(s.values, n_classes=k)
    except TypeError:
        breaks = jenkspy.jenks_breaks(s.values, nb_class=k)
    breaks = sorted(set(float(b) for b in breaks))
    if len(breaks) < 3:
        return pd.Series('L1', index=v.index, dtype=object)
    breaks[0], breaks[-1] = -np.inf, np.inf
    labels = [f"L{i+1}" for i in range(len(breaks) - 1)]
    return pd.cut(v, bins=breaks, labels=labels, include_lowest=True).astype(str)

GEO_MAPPER_COLS = list(spatial_raw.columns)
spatial_df = pd.DataFrame(index=rbind_data.index)
for _col in GEO_MAPPER_COLS:
    spatial_df[_col] = jenks_bin_train(spatial_raw[_col], train_pos, n_classes=5)

print('train-only spatial Jenks 完成:', GEO_MAPPER_COLS)
display(spatial_df.apply(lambda s: s.value_counts().sort_index()))


In [ ]:
# ==== Mapper 基礎結果快取：不保存 betweenness / coreness 等衍生拓樸量 ====
MAPPER_CACHE_DIR = os.path.join(parent_dir, 'Models', 'MapperCache')
MAPPER_CACHE_PATH = os.path.join(MAPPER_CACHE_DIR, 'full_mapper_base.pkl')
REBUILD_MAPPER = False  # Mapper 設定或輸入欄位改變時，手動改 True 重建一次

_mapper_cache_loaded = os.path.exists(MAPPER_CACHE_PATH) and not REBUILD_MAPPER
if _mapper_cache_loaded:
    with open(MAPPER_CACHE_PATH, 'rb') as _f:
        _mapper_cache = pickle.load(_f)
    if _mapper_cache.get('n_rows') != _n:
        raise ValueError('Mapper cache row count 不符；請設 REBUILD_MAPPER=True')
    if not np.array_equal(_mapper_cache['train_pos'], train_pos):
        raise ValueError('Mapper cache train split 不符；請設 REBUILD_MAPPER=True')
    if not np.array_equal(_mapper_cache['test_pos'], test_pos):
        raise ValueError('Mapper cache OOS split 不符；請設 REBUILD_MAPPER=True')
    graph = _mapper_cache['graph']
    lens_train = _mapper_cache['lens_train']
    lens_test = _mapper_cache['lens_test']
    oos_source_pos = _mapper_cache['oos_source_pos']
    print('已讀取 Mapper base cache:', MAPPER_CACHE_PATH)
    print('cache 只含 graph/lens/OOS mapping；centrality 會在下一格重算')
else:
    # ==== 地理特徵加入點雲/MCA；所有表示只 fit train ====
    from utils.preprocess import PARTY_TIME_COLS
    from sklearn.neighbors import NearestNeighbors

    BASE_MAPPER_COLS = [c for c in rbind_data.columns if c not in PARTY_TIME_COLS]
    mapper_select_lst = BASE_MAPPER_COLS + GEO_MAPPER_COLS
    mapper_feature_df = pd.concat([rbind_data[BASE_MAPPER_COLS], spatial_df[GEO_MAPPER_COLS]], axis=1).astype(str)
    _FORBIDDEN_LABEL_COLS = {'死亡', '受傷', '死亡受傷人數', 'color_for_plot', 'y'}
    assert not (_FORBIDDEN_LABEL_COLS & set(mapper_feature_df.columns)), 'Mapper 輸入含標籤欄位'
    mapper_train_source = mapper_feature_df.iloc[train_pos]
    mapper_test_source = mapper_feature_df.iloc[test_pos]

    # one-hot vocabulary 與 MCA 都只由 train 決定
    mapper_train_df = pd.get_dummies(mapper_train_source, dtype=np.float32)
    mapper_test_df = pd.get_dummies(mapper_test_source, dtype=np.float32).reindex(columns=mapper_train_df.columns, fill_value=0)
    mapper_train = mapper_train_df.to_numpy()
    mapper_test = mapper_test_df.to_numpy()

    mca = prince.MCA(
        n_components=2, n_iter=30, copy=True,
        check_input=True, random_state=42
    )
    mca.fit(mapper_train_source)
    lens_train = mca.transform(mapper_train_source).to_numpy()
    lens_test = mca.transform(mapper_test_source).to_numpy()

    _unseen = {
        c: int((~mapper_test_source[c].isin(mapper_train_source[c].unique())).sum())
        for c in mapper_select_lst
    }
    _unseen = {c: n for c, n in _unseen.items() if n > 0}
    print('Mapper 欄位數:', len(mapper_select_lst), '| point cloud:', mapper_train.shape, mapper_test.shape)
    print('MCA lens:', lens_train.shape, lens_test.shape, '| test-only 類別:', _unseen or '無')


In [ ]:
overlap = 2
interval = 10

if not _mapper_cache_loaded:
    mapper_algo = MapperAlgorithm(
        cover=CubicalCover(n_intervals=interval, overlap_frac=overlap / 10),
        clustering=FailSafeClustering(
            AgglomerativeClustering(n_clusters=2, linkage='ward')),
        n_jobs=10)
    _res = mapper_algo.fit_transform(mapper_train, lens_train)
    graph = _res[0] if isinstance(_res, tuple) else _res
    if not isinstance(graph, nx.Graph):
        graph = getattr(mapper_algo, 'graph_', graph)
        if isinstance(graph, tuple):
            graph = graph[0]

nodes = list(graph.nodes())
if not nodes:
    raise ValueError('Mapper 沒有產生任何 node')
print('train Mapper：nodes =', len(nodes), '| edges =', graph.number_of_edges(),
      '| source =', 'cache' if _mapper_cache_loaded else 'new fit')

# ==== train graph 上的 node-level 拓樸量；完全不使用死亡標籤 ====
node_size = {nd: len(graph.nodes[nd]['ids']) for nd in nodes}
_all_local_ids = np.concatenate([
    np.asarray(graph.nodes[nd]['ids'], dtype=int) for nd in nodes
])
assert _all_local_ids.min() >= 0 and _all_local_ids.max() < len(train_pos)

# cycle_member：屬於至少一個含 3+ nodes 的 biconnected component。
# 這比 2-core 更嚴格，不會把連接兩個 cycles 的 bridge path 誤標成 cycle。
_cycle_nodes = set()
for _block in nx.biconnected_components(graph):
    if len(_block) >= 3:
        _cycle_nodes.update(_block)
node_cycle_member = {nd: int(nd in _cycle_nodes) for nd in nodes}

node_coreness = nx.core_number(graph)
node_betweenness = nx.betweenness_centrality(graph, normalized=True, weight=None)
node_clustering = nx.clustering(graph, weight=None)
_articulation_nodes = set(nx.articulation_points(graph))
node_articulation = {nd: int(nd in _articulation_nodes) for nd in nodes}
_bridge_edges = list(nx.bridges(graph))
_bridge_nodes = {nd for edge in _bridge_edges for nd in edge}
node_bridge_endpoint = {nd: int(nd in _bridge_nodes) for nd in nodes}

def _node_distance_to_sources(sources):
    """Unweighted graph distance；無法到達來源的 component 使用 max(finite)+1。"""
    if not sources:
        return {nd: 1.0 for nd in nodes}
    finite = nx.multi_source_dijkstra_path_length(graph, sources, weight=None)
    unreachable_value = float(max(finite.values(), default=0) + 1)
    return {nd: float(finite.get(nd, unreachable_value)) for nd in nodes}

node_distance_to_cycle = _node_distance_to_sources(_cycle_nodes)

# main core：含最大 observation node 的 component 中，coreness 最高的 nodes。
_anchor = max(nodes, key=lambda nd: node_size[nd])
_main_component = set(nx.node_connected_component(graph, _anchor))
_main_core_k = max(node_coreness[nd] for nd in _main_component)
_main_core_nodes = {nd for nd in _main_component if node_coreness[nd] == _main_core_k}
node_distance_to_main_core = _node_distance_to_sources(_main_core_nodes)

print('cycle nodes:', len(_cycle_nodes),
      '| articulation nodes:', len(_articulation_nodes),
      '| bridges:', len(_bridge_edges),
      '| main core k:', _main_core_k, '| main core nodes:', len(_main_core_nodes))

# ==== node → train observation 聚合 ====
# binary=any；distance=min；coreness/betweenness=max；clustering=membership mean。
topo_cycle_member = np.zeros(_n, dtype=np.int8)
topo_distance_to_cycle = np.full(_n, np.inf)
topo_coreness = np.full(_n, -np.inf)
topo_betweenness = np.full(_n, -np.inf)
topo_clustering_sum = np.zeros(_n, dtype=np.float64)
topo_clustering_coefficient = np.full(_n, np.nan)
topo_articulation_point = np.zeros(_n, dtype=np.int8)
topo_bridge_endpoint = np.zeros(_n, dtype=np.int8)
topo_distance_to_main_core = np.full(_n, np.inf)
membership_count = np.zeros(_n, dtype=np.int16)

for nd in nodes:
    loc = np.asarray(graph.nodes[nd]['ids'], dtype=int)  # train-local positions
    orig = train_pos[loc]                                # original/global positions
    topo_cycle_member[orig] = np.maximum(topo_cycle_member[orig], node_cycle_member[nd])
    topo_distance_to_cycle[orig] = np.minimum(topo_distance_to_cycle[orig], node_distance_to_cycle[nd])
    topo_coreness[orig] = np.maximum(topo_coreness[orig], node_coreness[nd])
    topo_betweenness[orig] = np.maximum(topo_betweenness[orig], node_betweenness[nd])
    topo_clustering_sum[orig] += node_clustering[nd]
    topo_articulation_point[orig] = np.maximum(topo_articulation_point[orig], node_articulation[nd])
    topo_bridge_endpoint[orig] = np.maximum(topo_bridge_endpoint[orig], node_bridge_endpoint[nd])
    topo_distance_to_main_core[orig] = np.minimum(topo_distance_to_main_core[orig], node_distance_to_main_core[nd])
    membership_count[orig] += 1

topo_clustering_coefficient[train_pos] = (topo_clustering_sum[train_pos] / membership_count[train_pos])

# ==== OOS → train 對應：只在第一次建圖時計算，之後直接讀小型 mapping ====
if not _mapper_cache_loaded:
    from sklearn.neighbors import NearestNeighbors

    CANDIDATE_K = min(100, len(train_pos))
    ASSIGN_CHUNK = 1000
    _lens_nn = NearestNeighbors(n_neighbors=CANDIDATE_K, algorithm='kd_tree', metric='euclidean', n_jobs=-1).fit(lens_train)
    oos_source_pos = np.full(_n, -1, dtype=np.int64)
    for start in range(0, len(test_pos), ASSIGN_CHUNK):
        stop = min(start + ASSIGN_CHUNK, len(test_pos))
        candidates = _lens_nn.kneighbors(lens_test[start:stop], return_distance=False)
        point_dist = np.count_nonzero(mapper_train[candidates] != mapper_test[start:stop, None, :], axis=2,)
        nearest_local = candidates[np.arange(stop - start), point_dist.argmin(axis=1)]
        target_pos = test_pos[start:stop]
        oos_source_pos[target_pos] = train_pos[nearest_local]

    if (oos_source_pos[test_pos] < 0).any():
        raise ValueError('部分 OOS observations 沒有 train 對應')

    # graph 只存 node membership/edges；betweenness 等仍是上方每次由 graph 重算。
    os.makedirs(MAPPER_CACHE_DIR, exist_ok=True)
    _mapper_cache = {
        'cache_version': 1,
        'n_rows': int(_n),
        'train_pos': np.asarray(train_pos, dtype=np.int64),
        'test_pos': np.asarray(test_pos, dtype=np.int64),
        'graph': graph,
        'lens_train': np.asarray(lens_train, dtype=np.float32),
        'lens_test': np.asarray(lens_test, dtype=np.float32),
        'oos_source_pos': oos_source_pos,
        'mapper_parameters': {
            'interval': int(interval),
            'overlap': float(overlap / 10),
            'mca_components': int(lens_train.shape[1]),
        },
        'explicitly_not_cached': [
            'betweenness', 'coreness', 'clustering', 'cycle_member',
            'articulation_point', 'bridge_endpoint',
            'distance_to_cycle', 'distance_to_main_core', 'topology_df',
        ],
    }
    with open(MAPPER_CACHE_PATH, 'wb') as _f:
        pickle.dump(_mapper_cache, _f, protocol=pickle.HIGHEST_PROTOCOL)
    print('已保存 Mapper base cache:', MAPPER_CACHE_PATH)
else:
    CANDIDATE_K = 100  # 保留變數名稱，供舊的後續 cells 相容
    ASSIGN_CHUNK = 1000

# 由 train observation 的最新拓樸摘要傳給 OOS；不需要 mapper_train/mapper_test。
for start in range(0, len(test_pos), ASSIGN_CHUNK):
    stop = min(start + ASSIGN_CHUNK, len(test_pos))
    target_pos = test_pos[start:stop]
    source_pos = oos_source_pos[target_pos]
    topo_cycle_member[target_pos] = topo_cycle_member[source_pos]
    topo_distance_to_cycle[target_pos] = topo_distance_to_cycle[source_pos]
    topo_coreness[target_pos] = topo_coreness[source_pos]
    topo_betweenness[target_pos] = topo_betweenness[source_pos]
    topo_clustering_coefficient[target_pos] = topo_clustering_coefficient[source_pos]
    topo_articulation_point[target_pos] = topo_articulation_point[source_pos]
    topo_bridge_endpoint[target_pos] = topo_bridge_endpoint[source_pos]
    topo_distance_to_main_core[target_pos] = topo_distance_to_main_core[source_pos]


In [ ]:
# ==== 拓樸特徵直接使用原始數值；不做 Jenks ====
TOPOLOGY_COLUMNS = [
    'topo_cycle_member',
    'topo_distance_to_cycle',
    'topo_clustering_coefficient',
    'topo_articulation_point',
    'topo_bridge_endpoint',

    'topo_coreness',
    'topo_betweenness',
    'topo_distance_to_main_core',
]
topo_df = pd.DataFrame({
    'topo_cycle_member': topo_cycle_member,
    'topo_distance_to_cycle': topo_distance_to_cycle,
    'topo_clustering_coefficient': topo_clustering_coefficient,
    'topo_articulation_point': topo_articulation_point,
    'topo_bridge_endpoint': topo_bridge_endpoint,

    'topo_coreness': topo_coreness,
    'topo_betweenness': topo_betweenness,
    'topo_distance_to_main_core': topo_distance_to_main_core,
}, index=rbind_data.index)


In [ ]:
# ==== 合併回完整檔並依時間排序、存檔 ====
base = rbind_data.copy()
base['死亡'] = death.values
base = pd.concat([base, spatial_df], axis=1)
base['發生日期'] = date_info['發生日期'].values
base['發生時間'] = date_info['發生時間'].values

order = _order_index
base = base.loc[order].reset_index(drop=True)
topo = topo_df.loc[order].reset_index(drop=True)
full_topo = pd.concat([base, topo], axis=1)
full_notopo = base.copy()
topology_only = pd.concat([base[['死亡', '發生日期', '發生時間']], topo], axis=1)

os.makedirs('./Data/FullData', exist_ok=True)
full_topo.to_csv('./Data/FullData/full_with_topo.csv', index=False)
full_notopo.to_csv('./Data/FullData/full_no_topo.csv', index=False)
topology_only.to_csv('./Data/FullData/topology_only.csv', index=False)
print('已存檔:', full_topo.shape, full_notopo.shape, topology_only.shape)
print('topology columns:', topo.columns.tolist())


In [ ]:
# ==== 訓練：時間順序 70% train / 10% validation / 20% test ====
# 重新載入 models.py，確保目前 kernel 使用 validation-threshold 新版函式。
_spec.loader.exec_module(models_new)
DROP = [c for c in ['發生日期', '發生時間'] if c in full_topo.columns]
X_topo,   y_topo   = models_new.get_train_test_data(full_topo.drop(columns=DROP).copy())
X_notopo, y_notopo = models_new.get_train_test_data(full_notopo.drop(columns=DROP).copy())
X_topology_only, y_topology_only = models_new.get_train_test_data(topology_only.drop(columns=DROP).copy())
print('X_notopo:', X_notopo.shape,
      '| X_topo:', X_topo.shape,
      '| X_topology_only:', X_topology_only.shape)

SEED = 42
TRAIN_UNDER_RATIO = 1
ENCODING = 'onehot'
THRESHOLD_STRATEGY = 'youden'
RESULT_ROOT = '../Models/ModelPerformanceSeed/validation_youden_70_10_20'
_n_model = len(X_topo)
assert len(final_test_pos) == _n_model - len(train_pos) - len(validation_pos)
print('model split | train:', int(np.floor(_n_model * TRAIN_FRAC)),
      '| validation:', int(np.floor(_n_model * VALIDATION_FRAC)),
      '| test:', _n_model - int(np.floor(_n_model * TRAIN_FRAC))
                   - int(np.floor(_n_model * VALIDATION_FRAC)))
algos = [('xgboost', models_new.xgboost_cm_gridsearch),
         ('logistic', models_new.logistic_cm_gridsearch),
         ('svc', models_new.linear_svc_cm_gridsearch)]

def train_and_save_variant(tag, X, y, result_root):
    """固定 70/10/20，並在 validation 自動選擇 Youden's J 最大的 threshold。"""
    for algo_name, fn in algos:
        print(f'[{tag}] {algo_name} start')
        t = time.time()
        yv, sc, idx, threshold_info = fn(
            X, y,
            random_state=SEED,
            encoding=ENCODING,
            train_under_ratio=TRAIN_UNDER_RATIO,
            threshold_strategy=THRESHOLD_STRATEGY,
            train_frac=TRAIN_FRAC,
            validation_frac=VALIDATION_FRAC,
        )
        elapsed = time.time() - t
        save_dir = f"{result_root}/{tag}/{algo_name}"
        os.makedirs(save_dir, exist_ok=True)
        with open(f"{save_dir}/full.pkl", "wb") as f:
            pickle.dump({
                'y': yv,
                'decision_scores': sc,
                'indices': idx,
                'elapsed_time': elapsed,
                'selected_threshold': threshold_info['selected_threshold'],
                'threshold_selection': threshold_info,
                'features': X.columns.tolist(),
                'time_split': {
                    'train': TRAIN_FRAC,
                    'validation': VALIDATION_FRAC,
                    'test': 1 - TRAIN_FRAC - VALIDATION_FRAC,
                },
            }, f)
        print(
            f'[{tag}] {algo_name} done in {elapsed:.1f}s | '
            f"threshold={threshold_info['selected_threshold']:.4f} | "
            f"val J={threshold_info['validation_youden_j']:.4f} | "
            f"recall={threshold_info['validation_recall']:.4f} | "
            f"specificity={threshold_info['validation_specificity']:.4f} | "
            f"precision={threshold_info['validation_precision']:.4f} | "
            f"F1={threshold_info['validation_f1']:.4f}"
        )

MAIN_VARIANTS = {
    'full_notopo': (X_notopo, y_notopo),
    'full_topo': (X_topo, y_topo),
    'topology_only': (X_topology_only, y_topology_only),
}
for tag, (X, y) in MAIN_VARIANTS.items():
    train_and_save_variant(tag, X, y, RESULT_ROOT)


In [ ]:
# ==== 比較：各模型使用 validation 選出的 threshold；test 不參與選門檻 ====
_spec2.loader.exec_module(evaluate)
rows = []
threshold_rows = []
for algo_name, _ in algos:
    for tag in ['full_notopo', 'full_topo', 'topology_only']:
        path = f"{RESULT_ROOT}/{tag}/{algo_name}/full.pkl"
        d = evaluate.load_pkl(path)
        m = evaluate.metrics_from_pkl(path, threshold='saved')
        rows.append((f"{algo_name} | {tag}", m))
        info = d['threshold_selection']
        threshold_rows.append({
            'method': f'{algo_name} | {tag}',
            'threshold_strategy': info['threshold_strategy'],
            'selected_threshold': info['selected_threshold'],
            'validation_youden_j': info['validation_youden_j'],
            'validation_recall': info['validation_recall'],
            'validation_specificity': info['validation_specificity'],
            'validation_precision': info['validation_precision'],
            'validation_f1': info['validation_f1'],
            'n_validation': info['n_validation'],
            'n_validation_positive': info['n_validation_positive'],
        })

print('Test metrics（threshold 已在 validation 固定）')
display(evaluate.build_table(rows))
print('Validation threshold audit')
display(pd.DataFrame(threshold_rows).set_index('method').round(4))

## Ablation 1 — 三個 Topology 特徵的完整子集合

固定原始事故與地理特徵，只改變加入模型的 topology 子集合。使用相同的時間 70/10/20、分類器、下採樣，並在各自的 validation 上自動選擇 Youden's J 最大的 threshold。

- **C**：`topo_coreness`
- **B**：`topo_betweenness`
- **D**：`topo_distance_to_main_core`

`none` 與 `C+B+D` 直接重用主比較的 `full_notopo` / `full_topo`，其餘六組才重新訓練。

In [ ]:
from itertools import combinations

TOPOLOGY_ABBR = {
    'topo_coreness': 'C',
    'topo_betweenness': 'B',
    'topo_distance_to_main_core': 'D',
}
assert TOPOLOGY_COLUMNS == list(TOPOLOGY_ABBR), (
    '此 section 針對目前三個 topology 特徵；若 TOPOLOGY_COLUMNS 改變，請同步更新縮寫')

SUBSET_RESULT_ROOT = f'{RESULT_ROOT}/ablation_topology_subsets'
subset_specs = []
subset_inputs = {}
for r in range(len(TOPOLOGY_COLUMNS) + 1):
    for cols_tuple in combinations(TOPOLOGY_COLUMNS, r):
        cols = list(cols_tuple)
        tag = 'none' if not cols else '_'.join(TOPOLOGY_ABBR[c] for c in cols)
        label = 'none' if not cols else '+'.join(TOPOLOGY_ABBR[c] for c in cols)
        variant_df = full_notopo.copy()
        for col in cols:
            variant_df[col] = topo[col].to_numpy()
        X_variant, y_variant = models_new.get_train_test_data(
            variant_df.drop(columns=DROP).copy())
        expected_columns = X_notopo.columns.tolist() + cols
        assert X_variant.columns.tolist() == expected_columns
        assert y_variant.reset_index(drop=True).equals(y_notopo.reset_index(drop=True))
        subset_specs.append((tag, label, cols))
        subset_inputs[tag] = (X_variant, y_variant)

for tag, label, cols in subset_specs:
    if len(cols) in (0, len(TOPOLOGY_COLUMNS)):
        print(f'[{label}] reuse main comparison result')
        continue
    train_and_save_variant(tag, *subset_inputs[tag], SUBSET_RESULT_ROOT)


In [ ]:
subset_validation_rows = []
for algo_name, _ in algos:
    subset_rows = []
    for tag, label, cols in subset_specs:
        if not cols:
            path = f'{RESULT_ROOT}/full_notopo/{algo_name}/full.pkl'
        elif len(cols) == len(TOPOLOGY_COLUMNS):
            path = f'{RESULT_ROOT}/full_topo/{algo_name}/full.pkl'
        else:
            path = f'{SUBSET_RESULT_ROOT}/{tag}/{algo_name}/full.pkl'
        d = evaluate.load_pkl(path)
        m = evaluate.metrics_from_pkl(path, threshold='saved')
        method = f'{algo_name} | {label}'
        subset_rows.append((method, m))
        info = d['threshold_selection']
        subset_validation_rows.append({
            'method': method,
            'selected_threshold': info['selected_threshold'],
            'validation_youden_j': info['validation_youden_j'],
            'validation_recall': info['validation_recall'],
            'validation_specificity': info['validation_specificity'],
            'validation_precision': info['validation_precision'],
            'validation_f1': info['validation_f1'],
        })
    print(f'Topology subset ablation | {algo_name} | final test')
    display(evaluate.build_table(subset_rows))

print('Topology subset validation threshold audit')
display(pd.DataFrame(subset_validation_rows).set_index('method').round(4))


## Ablation 2 — MCA 表示與 Mapper Topology

比較相同 MCA 來源下的三種模型輸入：

1. **MCA-only**：只使用 train-fitted MCA 的 5 維座標。
2. **Topology-only**：只使用 Mapper graph 提取的三個 topology 特徵，重用主比較結果。
3. **MCA + Topology**：同時使用 MCA 座標與 topology 特徵。

MCA train/OOS rows 會先放回原始 observation index，再依共同時間順序排列，避免 row mismatch。

In [ ]:
MCA_DIM = lens_train.shape[1]
mca_all = np.full((_n, MCA_DIM), np.nan, dtype=np.float64)
mca_all[train_pos] = np.asarray(lens_train, dtype=np.float64)
mca_all[test_pos] = np.asarray(lens_test, dtype=np.float64)
mca_df = pd.DataFrame(
    mca_all,
    index=rbind_data.index,
    columns=[f'mca_{i + 1}' for i in range(MCA_DIM)],
).loc[order].reset_index(drop=True)

model_meta = base[['死亡', '發生日期', '發生時間']].copy()
mca_only_df = pd.concat([model_meta, mca_df], axis=1)
mca_topology_df = pd.concat([model_meta, mca_df, topo], axis=1)
X_mca_only, y_mca_only = models_new.get_train_test_data(
    mca_only_df.drop(columns=DROP).copy())
X_mca_topology, y_mca_topology = models_new.get_train_test_data(
    mca_topology_df.drop(columns=DROP).copy())

MECHANISM_RESULT_ROOT = f'{RESULT_ROOT}/ablation_mca_topology'
train_and_save_variant('mca_only', X_mca_only, y_mca_only, MECHANISM_RESULT_ROOT)
train_and_save_variant('mca_plus_topology', X_mca_topology, y_mca_topology, MECHANISM_RESULT_ROOT)


In [ ]:
mechanism_specs = [
    ('MCA-only', MECHANISM_RESULT_ROOT, 'mca_only'),
    ('Topology-only', RESULT_ROOT, 'topology_only'),
    ('MCA + Topology', MECHANISM_RESULT_ROOT, 'mca_plus_topology'),
]
mechanism_validation_rows = []
for algo_name, _ in algos:
    mechanism_rows = []
    for label, root, tag in mechanism_specs:
        path = f'{root}/{tag}/{algo_name}/full.pkl'
        d = evaluate.load_pkl(path)
        m = evaluate.metrics_from_pkl(path, threshold='saved')
        method = f'{algo_name} | {label}'
        mechanism_rows.append((method, m))
        info = d['threshold_selection']
        mechanism_validation_rows.append({
            'method': method,
            'selected_threshold': info['selected_threshold'],
            'validation_youden_j': info['validation_youden_j'],
            'validation_recall': info['validation_recall'],
            'validation_specificity': info['validation_specificity'],
            'validation_precision': info['validation_precision'],
            'validation_f1': info['validation_f1'],
        })
    print(f'MCA / Topology mechanism ablation | {algo_name} | final test')
    display(evaluate.build_table(mechanism_rows))

print('MCA / Topology validation threshold audit')
display(pd.DataFrame(mechanism_validation_rows).set_index('method').round(4))


## Ablation 3 — Train-only Mapper Node Fatality Risk

在固定的 train Mapper graph 上，以每個 node 內的死亡比例建立 supervised topology risk。為避免 training observation 的標籤直接出現在自己的特徵中，train rows 使用 leave-one-out node risk；validation/test 則只使用完整 train labels 得到的 node risk，再沿用既有 OOS 最近 train observation 指派。

小型 nodes 使用 global train fatality rate 做 Beta-style smoothing：`risk = (node_deaths + alpha * global_rate) / (node_size + alpha)`。Mapper graph、MCA、OOS 指派與 threshold selection 均維持原設定，validation/test labels 不參與 risk 建立。

In [ ]:
NODE_RISK_PRIOR_STRENGTH = 50.0
_death_binary = (np.asarray(death).reshape(-1) >= 1).astype(np.int8)
assert len(_death_binary) == _n
_train_y_local = _death_binary[train_pos]
_n_train_risk = len(train_pos)
_total_train_deaths = int(_train_y_local.sum())
_global_train_fatality_rate = _total_train_deaths / _n_train_risk

# 每個 observation 可能因 cover overlap 屬於多個 nodes；最後取 memberships 平均。
_risk_train_loo_sum = np.zeros(_n, dtype=np.float64)
_risk_train_full_sum = np.zeros(_n, dtype=np.float64)
_risk_membership_count = np.zeros(_n, dtype=np.int16)
_node_risk_records = []
_node_fatality_risk_full = {}

for _risk_node in nodes:
    _risk_local_ids = np.asarray(
        graph.nodes[_risk_node]['ids'], dtype=int)
    if len(_risk_local_ids) == 0:
        continue
    _risk_original_positions = train_pos[_risk_local_ids]
    _risk_member_y = _train_y_local[_risk_local_ids].astype(np.float64)
    _risk_node_size = int(len(_risk_local_ids))
    _risk_node_deaths = float(_risk_member_y.sum())
    _risk_node_raw = _risk_node_deaths / _risk_node_size
    _risk_node_smoothed = (
        _risk_node_deaths
        + NODE_RISK_PRIOR_STRENGTH * _global_train_fatality_rate
    ) / (_risk_node_size + NODE_RISK_PRIOR_STRENGTH)
    _node_fatality_risk_full[_risk_node] = float(_risk_node_smoothed)
    graph.nodes[_risk_node]['train_fatality_count'] = int(
        _risk_node_deaths)
    graph.nodes[_risk_node]['train_fatality_ratio_raw'] = float(
        _risk_node_raw)
    graph.nodes[_risk_node]['train_fatality_risk_smoothed'] = float(
        _risk_node_smoothed)

    # Strict self-exclusion：每個 train row 的 node deaths/size 都扣除自己。
    # global prior 也扣除該 row，避免標籤經由 prior 回流。
    _risk_global_loo = (
        (_total_train_deaths - _risk_member_y)
        / max(_n_train_risk - 1, 1)
    )
    _risk_node_loo = (
        (_risk_node_deaths - _risk_member_y)
        + NODE_RISK_PRIOR_STRENGTH * _risk_global_loo
    ) / (
        max(_risk_node_size - 1, 0)
        + NODE_RISK_PRIOR_STRENGTH
    )

    _risk_train_loo_sum[_risk_original_positions] += _risk_node_loo
    _risk_train_full_sum[_risk_original_positions] += _risk_node_smoothed
    _risk_membership_count[_risk_original_positions] += 1
    _node_risk_records.append({
        'node': _risk_node,
        'size': _risk_node_size,
        'deaths': int(_risk_node_deaths),
        'raw_fatality_ratio': float(_risk_node_raw),
        'smoothed_fatality_risk': float(_risk_node_smoothed),
    })

if (_risk_membership_count[train_pos] == 0).any():
    raise ValueError('部分 train observations 沒有 Mapper risk membership')
_risk_train_loo = np.full(_n, np.nan, dtype=np.float64)
_risk_train_full = np.full(_n, np.nan, dtype=np.float64)
_risk_train_loo[train_pos] = (
    _risk_train_loo_sum[train_pos]
    / _risk_membership_count[train_pos]
)
_risk_train_full[train_pos] = (
    _risk_train_full_sum[train_pos]
    / _risk_membership_count[train_pos]
)

# OOS 不使用自身標籤：直接重用 Mapper base cache 的固定 train 對應。
_topo_node_fatality_risk = _risk_train_loo.copy()
_risk_oos_source = np.full(_n, -1, dtype=np.int64)
_risk_oos_source[test_pos] = oos_source_pos[test_pos]
_topo_node_fatality_risk[test_pos] = _risk_train_full[oos_source_pos[test_pos]]

assert (_risk_oos_source[test_pos] >= 0).all()
assert np.isfinite(_topo_node_fatality_risk).all()
assert ((_topo_node_fatality_risk >= 0)
        & (_topo_node_fatality_risk <= 1)).all()

node_fatality_risk_df = pd.DataFrame(_node_risk_records).sort_values(
    ['smoothed_fatality_risk', 'size'], ascending=[False, False]
).reset_index(drop=True)
NODE_RISK_COLUMNS = ['topo_node_fatality_risk']
node_risk_df = pd.DataFrame({
    'topo_node_fatality_risk': _topo_node_fatality_risk,
}, index=rbind_data.index)
node_risk = node_risk_df.loc[order].reset_index(drop=True)
assert len(node_risk) == len(base) == len(topo)

model_meta = base[['死亡', '發生日期', '發生時間']].copy()
risk_only_df = pd.concat([model_meta, node_risk], axis=1)
full_risk_df = pd.concat([full_notopo, node_risk], axis=1)
full_topo_risk_df = pd.concat([full_topo, node_risk], axis=1)

X_risk_only, y_risk_only = models_new.get_train_test_data(
    risk_only_df.drop(columns=DROP).copy())
X_full_risk, y_full_risk = models_new.get_train_test_data(
    full_risk_df.drop(columns=DROP).copy())
X_full_topo_risk, y_full_topo_risk = models_new.get_train_test_data(
    full_topo_risk_df.drop(columns=DROP).copy())

assert X_risk_only.columns.tolist() == NODE_RISK_COLUMNS
assert X_full_risk.columns.tolist() == (
    X_notopo.columns.tolist() + NODE_RISK_COLUMNS)
assert X_full_topo_risk.columns.tolist() == (
    X_topo.columns.tolist() + NODE_RISK_COLUMNS)
for _risk_y_variant in [y_risk_only, y_full_risk, y_full_topo_risk]:
    assert _risk_y_variant.reset_index(drop=True).equals(
        y_notopo.reset_index(drop=True))

NODE_RISK_RESULT_ROOT = (
    f'{RESULT_ROOT}/ablation_mapper_node_risk_loo_alpha50')
NODE_RISK_VARIANTS = {
    'risk_only': (X_risk_only, y_risk_only),
    'full_risk': (X_full_risk, y_full_risk),
    'full_topo_risk': (X_full_topo_risk, y_full_topo_risk),
}
print('global train fatality rate:', round(_global_train_fatality_rate, 6),
      '| nodes:', len(node_fatality_risk_df),
      '| alpha:', NODE_RISK_PRIOR_STRENGTH)
display(node_fatality_risk_df.head(10))
print('risk variant shapes:', {
    tag: X.shape for tag, (X, _) in NODE_RISK_VARIANTS.items()})
for _risk_tag, (_risk_X, _risk_y) in NODE_RISK_VARIANTS.items():
    train_and_save_variant(
        _risk_tag, _risk_X, _risk_y, NODE_RISK_RESULT_ROOT)


In [ ]:
node_risk_specs = [
    ('No topology', RESULT_ROOT, 'full_notopo'),
    ('Current centrality', RESULT_ROOT, 'full_topo'),
    ('Current topology-only', RESULT_ROOT, 'topology_only'),
    ('Mapper risk-only', NODE_RISK_RESULT_ROOT, 'risk_only'),
    ('Base + Mapper risk', NODE_RISK_RESULT_ROOT, 'full_risk'),
    ('Base + Centrality + Mapper risk', NODE_RISK_RESULT_ROOT, 'full_topo_risk'),
]
node_risk_validation_rows = []
for algo_name, _ in algos:
    node_risk_rows = []
    for label, root, tag in node_risk_specs:
        path = f'{root}/{tag}/{algo_name}/full.pkl'
        d = evaluate.load_pkl(path)
        m = evaluate.metrics_from_pkl(path, threshold='saved')
        method = f'{algo_name} | {label}'
        node_risk_rows.append((method, m))
        info = d['threshold_selection']
        node_risk_validation_rows.append({
            'method': method,
            'selected_threshold': info['selected_threshold'],
            'validation_youden_j': info['validation_youden_j'],
            'validation_recall': info['validation_recall'],
            'validation_specificity': info['validation_specificity'],
            'validation_precision': info['validation_precision'],
            'validation_f1': info['validation_f1'],
        })
    print(f'Mapper node fatality risk | {algo_name} | final test')
    display(evaluate.build_table(node_risk_rows))

print('Mapper node risk validation threshold audit')
display(pd.DataFrame(
    node_risk_validation_rows).set_index('method').round(4))
